In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print(" Setup complete —", llm.invoke("say ok").content)

 Setup complete — Ok.


In [2]:
from typing import TypedDict, Annotated, List, Dict, Optional
from operator import add

class ResearchState(TypedDict):
    query: str
    plan: List[str]
    needs_more_research: bool
    search_results: Annotated[List[Dict], add]
    extracted_facts: Annotated[List[Dict], add]
    fact_check_results: Annotated[List[Dict], add]
    summaries: Annotated[List[Dict], add]
    final_report: Optional[str]
    agent_log: Annotated[List[str], add]

print(" ResearchState re-defined (added 'needs_more_research' for today's routing demo)")

 ResearchState re-defined (added 'needs_more_research' for today's routing demo)


In [1]:
def planner_node(state: ResearchState) -> dict:
    print("Planner running...")
    prompt = f"""Break this research question into sub-questions: {state['query']}
Reply with each sub-question on its own line, nothing else."""
    response = llm.invoke(prompt)
    sub_questions = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip()]

    
    classify_prompt = f"""Is this research query specific enough to research directly, or is it too
vague/broad (like a single word or generic topic)? Query: "{state['query']}"
Reply with exactly one word: SPECIFIC or VAGUE."""
    classification = llm.invoke(classify_prompt).content.strip().upper()
    needs_more = "VAGUE" in classification

    return {
        "plan": sub_questions,
        "needs_more_research": needs_more,
        "agent_log": [f"planner_node: generated {len(sub_questions)} sub-questions, classified query as {classification}"]
    }

NameError: name 'ResearchState' is not defined

In [4]:
def route_after_planning(state: ResearchState) -> str:
    """Returns the NAME of the next node to go to. This function doesn't touch state,
    it just reads it and decides where the graph should go next."""
    if state["needs_more_research"]:
        print("Router: plan too thin → routing to 'clarify'")
        return "clarify"
    else:
        print(" Router: plan looks solid → routing to 'researcher'")
        return "researcher"

print("route_after_planning defined")

route_after_planning defined


In [5]:
def researcher_node(state: ResearchState) -> dict:
    print(" Researcher running...")
    return {
        "search_results": [{"title": "dummy result", "url": "https://example.com"}],
        "agent_log": ["researcher_node: ran search on all sub-questions"]
    }

def clarify_node(state: ResearchState) -> dict:
    print(" Clarify running (plan was too vague)...")
    return {
        "agent_log": ["clarify_node: flagged query as needing clarification before research"]
    }

print(" researcher_node and clarify_node defined")

 researcher_node and clarify_node defined


In [9]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ResearchState)

graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("clarify", clarify_node)

graph.add_edge(START, "planner")


graph.add_conditional_edges(
    "planner",
    route_after_planning,
    {
        "researcher": "researcher",
        "clarify": "clarify"
    }
)

graph.add_edge("researcher", END)
graph.add_edge("clarify", END)

app = graph.compile()
print(" Graph compiled with conditional routing")


 Graph compiled with conditional routing


In [10]:
result = app.invoke({
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL, including raw material sourcing, energy density, and manufacturing cost",
    "plan": [], "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "final_report": None, "agent_log": []
})
print("\nRoute taken — check agent_log:", result["agent_log"])

 Planner running...
 Router: plan looks solid → routing to 'researcher'
 Researcher running...

Route taken — check agent_log: ['planner_node: generated 9 sub-questions', 'researcher_node: ran search on all sub-questions']


In [11]:
result = app.invoke({
    "query": "batteries",
    "plan": [], "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "final_report": None, "agent_log": []
})
print("\nRoute taken — check agent_log:", result["agent_log"])

 Planner running...
 Router: plan looks solid → routing to 'researcher'
 Researcher running...

Route taken — check agent_log: ['planner_node: generated 10 sub-questions', 'researcher_node: ran search on all sub-questions']


In [13]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
app_with_memory = graph.compile(checkpointer=checkpointer)

# a "thread_id" identifies one continuous session/run — like a conversation ID
config = {"configurable": {"thread_id": "research-session-1"}}

result = app_with_memory.invoke({
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL",
    "plan": [], "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "final_report": None, "agent_log": []
}, config=config)

print(" Ran with checkpointer, thread_id = research-session-1")

 Planner running...
 Router: plan looks solid → routing to 'researcher'
 Researcher running...
 Ran with checkpointer, thread_id = research-session-1


In [14]:
saved_state = app_with_memory.get_state(config)

print("Current values:", saved_state.values["agent_log"])
print("Next node to run (empty = finished):", saved_state.next)

print("\n--- Full history of this thread ---")
for snapshot in app_with_memory.get_state_history(config):
    print(snapshot.next, "|", snapshot.values.get("agent_log"))

Current values: ['planner_node: generated 10 sub-questions', 'researcher_node: ran search on all sub-questions']
Next node to run (empty = finished): ()

--- Full history of this thread ---
() | ['planner_node: generated 10 sub-questions', 'researcher_node: ran search on all sub-questions']
('researcher',) | ['planner_node: generated 10 sub-questions']
('planner',) | []
('__start__',) | []


In [15]:
app_interruptible = graph.compile(checkpointer=checkpointer, interrupt_before=["researcher"])

config2 = {"configurable": {"thread_id": "research-session-2"}}

result = app_interruptible.invoke({
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL, including cost and supply chain",
    "plan": [], "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "final_report": None, "agent_log": []
}, config=config2)

state_now = app_interruptible.get_state(config2)
print("Paused before running:", state_now.next)
print("Plan generated so far:", state_now.values["plan"])

result = app_interruptible.invoke(None, config=config2)
print("\nAfter resuming:", result["agent_log"])

 Planner running...
 Router: plan looks solid → routing to 'researcher'
Paused before running: ('researcher',)
Plan generated so far: ['What are the primary battery chemistries used by Tesla, BYD, and CATL?', 'How do the cost structures of battery packs differ among Tesla, BYD, and CATL?', "What are the main sources of raw materials for each company's battery supply chain?", 'How does each company address raw material sourcing risks and geopolitical factors?', 'What are the manufacturing capacities and geographic locations of Tesla, BYD, and CATL battery production?', "How do partnerships, joint ventures, and supplier relationships influence each company's battery strategy?", "What are the projected cost trends for each company's battery technology over the next five years?", 'How do sustainability, recycling, and environmental initiatives compare across Tesla, BYD, and CATL?']
 Researcher running...

After resuming: ['planner_node: generated 8 sub-questions', 'researcher_node: ran sea